
## Count Rows


In [0]:
silver_sales = spark.table("sales_lakehouse.silver.sales")
gold_sales = spark.table("sales_lakehouse.gold.fact_sales")

print("Silver sales rows:", silver_sales.count())
print("Gold fact_sales rows:", gold_sales.count())

In [0]:
spark.table("sales_lakehouse.gold.fact_sales") \
    .filter("date_key IS NULL") \
    .count()

## Orphan Checks


In [0]:
display(spark.sql("""
SELECT COUNT(*) 
FROM sales_lakehouse.gold.fact_sales f
LEFT JOIN sales_lakehouse.gold.dim_product p
ON f.product_key = p.product_key
WHERE p.product_key IS NULL
"""))

In [0]:
display(spark.sql("""
SELECT COUNT(*) 
FROM sales_lakehouse.gold.fact_sales f
LEFT JOIN sales_lakehouse.gold.dim_reseller r
ON f.reseller_key = r.reseller_key
WHERE r.reseller_key IS NULL
"""))

In [0]:
display(spark.sql("""
SELECT COUNT(*) 
FROM sales_lakehouse.gold.fact_sales f
LEFT JOIN sales_lakehouse.gold.dim_salesperson s
ON f.employee_key = s.employee_key
WHERE s.employee_key IS NULL
"""))

In [0]:
display(spark.sql("""
SELECT COUNT(*) 
FROM sales_lakehouse.gold.fact_sales f
LEFT JOIN sales_lakehouse.gold.dim_region r
ON f.sales_territory_key = r.sales_territory_key
WHERE r.sales_territory_key IS NULL
"""))

## Grain Check


In [0]:
display(spark.sql("""
SELECT sales_order_number, product_key, COUNT(*)
FROM sales_lakehouse.gold.fact_sales
GROUP BY sales_order_number, product_key
HAVING COUNT(*) > 1
"""))

##   Targets - SalesPerson Connection

In [0]:
display(spark.sql("""
SELECT COUNT(*)
FROM sales_lakehouse.gold.fact_targets t
LEFT JOIN sales_lakehouse.gold.dim_salesperson s
ON t.employee_key = s.employee_key
WHERE s.employee_key IS NULL
"""))

## Join Tests


In [0]:
display(spark.sql("""
SELECT
p.product,
r.reseller_name,
s.salesperson_name,
d.year,
SUM(f.sales) AS total_sales
FROM sales_lakehouse.gold.fact_sales f
JOIN sales_lakehouse.gold.dim_product p
  ON f.product_key = p.product_key
JOIN sales_lakehouse.gold.dim_reseller r
  ON f.reseller_key = r.reseller_key
JOIN sales_lakehouse.gold.dim_salesperson s
  ON f.employee_key = s.employee_key
JOIN sales_lakehouse.gold.dim_date d
  ON f.date_key = d.date_key
GROUP BY
p.product,
r.reseller_name,
s.salesperson_name,
d.year
"""))

## Sales and Cost over time

In [0]:
display(spark.sql("""
SELECT
  d.year,
  d.month,
  d.month_name,
  SUM(f.sales) AS total_sales,
  SUM(f.cost) AS total_cost,
  SUM(f.sales) - SUM(f.cost) AS gross_profit
FROM sales_lakehouse.gold.fact_sales f
JOIN sales_lakehouse.gold.dim_date d
  ON f.date_key = d.date_key
GROUP BY
  d.year,
  d.month,
  d.month_name
ORDER BY
  d.year,
  d.month
"""))

## Sales By Region

In [0]:
display(spark.sql("""
SELECT
  r.region,
  SUM(f.sales) AS total_sales,
  SUM(f.cost) AS total_cost
FROM sales_lakehouse.gold.fact_sales f
JOIN sales_lakehouse.gold.dim_region r
  ON f.sales_territory_key = r.sales_territory_key
GROUP BY r.region
ORDER BY total_sales DESC
"""))

## Sales by salesperson

In [0]:
display(spark.sql("""
SELECT
  s.salesperson_name,
  SUM(f.sales) AS total_sales,
  SUM(f.cost) AS total_cost
FROM sales_lakehouse.gold.fact_sales f
JOIN sales_lakehouse.gold.dim_salesperson s
  ON f.employee_key = s.employee_key
GROUP BY s.salesperson_name
ORDER BY total_sales DESC
"""))

## Performance VS Targets

In [0]:
display(spark.sql("""
WITH sales_m AS (
  SELECT
    d.year_month,
    f.employee_key,
    SUM(f.sales) AS actual_sales
  FROM sales_lakehouse.gold.fact_sales f
  JOIN sales_lakehouse.gold.dim_date d
    ON f.date_key = d.date_key
  GROUP BY d.year_month, f.employee_key
),
targets_m AS (
  SELECT
    d.year_month,
    t.employee_key,
    SUM(t.target_sales_amount) AS target_sales
  FROM sales_lakehouse.gold.fact_targets t
  JOIN sales_lakehouse.gold.dim_date d
    ON t.date_key = d.date_key
  GROUP BY d.year_month, t.employee_key
)
SELECT
  s.year_month,
  sp.salesperson_name,
  s.actual_sales,
  t.target_sales,
  s.actual_sales - t.target_sales AS variance,
  ROUND(s.actual_sales / t.target_sales,5) AS attainment_pct
FROM sales_m s
JOIN targets_m t
  ON s.year_month = t.year_month
 AND s.employee_key = t.employee_key
JOIN sales_lakehouse.gold.dim_salesperson sp
  ON s.employee_key = sp.employee_key
ORDER BY s.year_month, sp.salesperson_name
"""))